# Aula de Ciência de Dados para Devs: Limpeza e Preparação de Dados

**Bem-vindo(a) à Parte 2: A Prática!**

Se na teoria tudo parece fazer sentido, é na prática que o conhecimento realmente se fixa. Nesta aula, você vai atuar como um detetive de dados. Vamos pegar um arquivo de cadastro de usuários de um e-commerce fictício (`usuarios_sujo.csv`), que está cheio de problemas comuns do mundo real, e vamos aplicar técnicas sistemáticas para limpá-lo e organizá-lo.

Ao final, teremos um dataset íntegro, confiável e pronto para a próxima fase: a análise exploratória de dados, onde poderemos extrair insights valiosos.

**Ferramentas que usaremos:**

- **Python:** Nossa linguagem de programação principal.
- **Pandas:** A biblioteca essencial para manipulação e análise de dados em Python. Pense nela como uma planilha superpoderosa que você controla com código.
- **NumPy:** Usada pelo Pandas por baixo dos panos, é fundamental para operações numéricas eficientes.
- **Faker:** Para gerar dados fictícios e criar nosso próprio dataset "sujo".


### Nosso Fluxo de Trabalho

Para nos guiar, seguiremos um fluxo de trabalho estruturado. Pense nisso como um mapa que nos levará do caos à ordem:

**1. Fonte de Dados Brutos (`usuarios_sujo.csv`)**

- Nosso ponto de partida. Um arquivo com dados realistas, porém problemáticos.

**2. Carregamento e Inspeção Inicial (Profiling)**

- Carregar os dados em um DataFrame e usar ferramentas de diagnóstico para identificar os problemas.

**3. Ciclo de Limpeza (Transformação)**

- Esta é a fase iterativa onde aplicamos as correções:
  - Tratar Valores Nulos
  - Corrigir Tipos de Dados
  - Remover Duplicatas
  - Padronizar Dados Categóricos

**4. Validação**

- Verificar se a limpeza foi bem-sucedida e se os dados agora fazem sentido.

**5. Saída de Dados Limpos (`usuarios_limpo.csv`)**

- Salvar nosso trabalho em um novo arquivo, pronto para ser usado em análises futuras.


---

## 1. Setup do Ambiente e Geração dos Dados

Primeiro, vamos importar as bibliotecas necessárias. Em seguida, vamos criar nosso próprio arquivo `usuarios_sujo.csv`. Isso garante que todos tenham exatamente o mesmo ponto de partida e que o notebook seja autocontido.

**Tipos de problemas que vamos introduzir:**

- **Valores Faltantes:** `email` e `valor_ultima_compra` terão valores nulos (`NaN`).
- **Tipos de Dados Incorretos:** `valor_ultima_compra` será uma string (object) com símbolos e vírgulas, e as colunas de data serão strings.
- **Formatos Inconsistentes:** A coluna `data_cadastro` terá múltiplos formatos de data (ex: 'YYYY-MM-DD', 'DD/MM/YYYY').
- **Dados Categóricos Não Padronizados:** A coluna `estado` terá variações como 'SP', 'São Paulo' e 'sao paulo'.
- **Linhas Duplicadas:** Inseriremos algumas linhas completamente duplicadas.


In [1]:
%pip install faker
# Importando as bibliotecas
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime

# Inicializando o Faker para gerar dados em português
fake = Faker('pt_BR')

# Função para gerar os dados sujos
def gerar_dados_sujos(num_usuarios=200):
    dados = []

    # Formatos de data que vamos misturar
    formatos_data = ['%Y-%m-%d', '%d/%m/%Y', '%m-%d-%Y', '%d-%b-%Y']

    # Variações para os estados
    estados_sp = ['SP', 'São Paulo', 'sao paulo']
    estados_rj = ['RJ', 'Rio de Janeiro', 'rio de janeiro']
    outros_estados = ['MG', 'PR', 'BA', 'SC']

    for i in range(num_usuarios):
        # Introduzindo valores faltantes (NaN) de forma aleatória
        email = fake.email() if random.random() > 0.1 else np.nan # 10% de chance de email nulo
        valor_compra = round(random.uniform(10, 1000), 2) if random.random() > 0.15 else np.nan # 15% de chance de valor nulo

        # Formatando o valor da compra como string com inconsistências
        if pd.notna(valor_compra):
            valor_compra_str = f"R$ {valor_compra:.2f}".replace('.', ',')
        else:
            # Adicionando outras strings não numéricas para sujar mais
            valor_compra_str = np.nan if random.random() > 0.3 else 'Não informado'

        # Escolhendo um formato de data aleatório
        formato_escolhido = random.choice(formatos_data)
        data_cadastro = fake.date_between(start_date='-2y', end_date='today').strftime(formato_escolhido)

        # Escolhendo um estado com inconsistências
        if i % 4 == 0:
            estado = random.choice(estados_sp)
        elif i % 7 == 0:
            estado = random.choice(estados_rj)
        else:
            estado = random.choice(outros_estados)

        dado = {
            'user_id': fake.uuid4(),
            'nome': fake.name(),
            'email': email,
            'data_cadastro': data_cadastro,
            'cidade': fake.city(),
            'estado': estado,
            'valor_ultima_compra': valor_compra_str,
            'data_ultimo_login': fake.date_time_between(start_date='-30d', end_date='now')
        }
        dados.append(dado)

    df = pd.DataFrame(dados)

    # Introduzindo linhas duplicadas
    duplicatas = df.sample(n=15, random_state=42)
    df_final = pd.concat([df, duplicatas]).reset_index(drop=True)

    return df_final

# Gerar e salvar o arquivo CSV
df_sujo = gerar_dados_sujos()
df_sujo.to_csv('usuarios_sujo.csv', index=False)

print("Arquivo 'usuarios_sujo.csv' gerado com sucesso!")
print(f"Total de linhas: {len(df_sujo)}")

/home/aldemir/Workspace/repositories/web-academy-ufam/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
Arquivo 'usuarios_sujo.csv' gerado com sucesso!
Total de linhas: 215


---

## 2. Passo a Passo do Exercício Prático

Agora começa a nossa missão! Vamos carregar o arquivo `usuarios_sujo.csv` que acabamos de criar e seguir nosso fluxo de trabalho para limpá-lo passo a passo.


### 2.1 Carregamento dos Dados

Usaremos a função `pd.read_csv()` do Pandas para ler nosso arquivo e carregá-lo em uma estrutura de dados chamada **DataFrame**. Pense no DataFrame como uma tabela ou planilha dentro do Python.


In [2]:
# Carregando o arquivo CSV para um DataFrame
df_usuarios = pd.read_csv('usuarios_sujo.csv')

# Exibindo as dimensões do DataFrame (linhas, colunas)
print(
    f"O dataset possui {df_usuarios.shape[0]} linhas e {df_usuarios.shape[1]} colunas.")

O dataset possui 215 linhas e 8 colunas.


### 2.2 Inspeção Inicial (Data Profiling)

Antes de sair corrigindo tudo, precisamos entender a "cena do crime". O Data Profiling é o processo de investigar o dataset para entender sua estrutura, qualidade e conteúdo. É aqui que identificamos os problemas.


#### Usando `.head()`, `.tail()` e `.sample()` para Amostragem

Essas funções nos permitem "espiar" os dados de diferentes ângulos:

- `.head(n)`: Mostra as primeiras `n` linhas (o padrão é 5).
- `.tail(n)`: Mostra as últimas `n` linhas (o padrão é 5).
- `.sample(n)`: Mostra uma amostra aleatória de `n` linhas, útil para ter uma visão imparcial do dataset.


In [3]:
print("--- Primeiras 5 linhas ---")
display(df_usuarios.head())

print("\n--- Últimas 5 linhas ---")
display(df_usuarios.tail())

print("\n--- Amostra aleatória de 5 linhas ---")
display(df_usuarios.sample(5))

--- Primeiras 5 linhas ---


,user_id,nome,email,data_cadastro,cidade,estado,valor_ultima_compra,data_ultimo_login
0,3e4cd4aa-904c-45d7-a242-cd4e64601acb,Ana Liz Novaes,nicolasgoncalves@example.net,24/06/2026,Dias,sao paulo,"R$ 968,82",2026-08-22 22:09:58.191486
1,6d4516ab-8cc3-47d2-9e43-f152ae798e00,Francisco Silveira,ybarros@example.net,01-Mar-2025,Albuquerque do Campo,PR,"R$ 275,88",2026-08-10 17:02:09.777518
2,fc579140-e00a-474a-8899-4f4fe4836ba5,Marcelo Viana,NaN,06/04/2025,Pinto,SC,"R$ 862,21",2026-08-16 13:59:36.257180
3,d02c239a-ca87-457d-9461-9fd94a15d9b5,Sabrina Abreu,qsilveira@example.com,30-Mar-2026,Ramos,SC,"R$ 754,25",2026-08-30 11:41:40.628894
4,fbb944bd-8db8-4454-96f0-c68163e37489,Luara Costela,NaN,02-07-2026,da Conceição de Nunes,São Paulo,NaN,2026-08-27 21:45:52.494628



--- Últimas 5 linhas ---


,user_id,nome,email,data_cadastro,cidade,estado,valor_ultima_compra,data_ultimo_login
210,62f81412-d3c5-4605-8235-50462e5618e5,Nicolas Moraes,NaN,16-Jun-2025,da Mota,SC,"R$ 475,66",2026-08-25 06:59:21.340266
211,07bd726e-6ea5-4048-9b6e-804b55ddc2be,Paulo da Rocha,cda-conceicao@example.net,05-Sep-2025,Caldeira dos Dourados,Rio de Janeiro,"R$ 28,73",2026-08-09 20:14:52.934201
212,1047fb7e-1171-4174-993e-7c8a66e89e0f,Emanuella Pires,bfreitas@example.com,2026-07-16,Pinto do Oeste,MG,"R$ 231,06",2026-08-01 20:56:15.095898
213,1b41faa8-f9ab-4e56-862e-4d55169b9296,Dr. Vinicius da Mata,joao-guilherme29@example.com,03-17-2025,Moura,SC,"R$ 117,68",2026-08-18 07:12:49.261655
214,8f086904-63f4-42e1-b2fa-8f064a3525c4,Sr. Juan Pereira,jose-pedro16@example.org,27-Jul-2026,Almeida dos Dourados,PR,"R$ 96,10",2026-08-01 00:03:49.115462



--- Amostra aleatória de 5 linhas ---


,user_id,nome,email,data_cadastro,cidade,estado,valor_ultima_compra,data_ultimo_login
202,d00597dc-12f9-4f2f-9939-14a0c20cd057,Gabrielly Cirino,cavalcantimaria@example.net,15/11/2025,Lopes do Sul,PR,"R$ 473,80",2026-08-19 23:14:55.653332
182,07bd726e-6ea5-4048-9b6e-804b55ddc2be,Paulo da Rocha,cda-conceicao@example.net,05-Sep-2025,Caldeira dos Dourados,Rio de Janeiro,"R$ 28,73",2026-08-09 20:14:52.934201
57,72d175fa-fd81-4510-b6df-0b007a941fa3,Dra. Alexia Viana,caioalmeida@example.net,2025-03-31,Aparecida,PR,"R$ 978,74",2026-08-24 23:32:14.361860
151,8a373bbf-99ee-4ff7-9090-5c6a76b26eb4,Juan Araújo,alexia65@example.net,09-15-2025,Aragão,PR,"R$ 531,82",2026-08-04 01:57:53.224395
38,6c411154-c4b3-4773-a03a-56a92bc1cc63,Maria Fernanda Nascimento,joao90@example.net,2024-10-02,Silveira,SC,"R$ 822,06",2026-08-29 02:36:19.275153


#### Usando `.info()` para um Resumo Técnico

O método `.info()` é um dos nossos melhores amigos. Ele nos dá um resumo conciso do DataFrame, incluindo:

- O número total de entradas (linhas).
- O número de colunas.
- O nome e a contagem de valores **não nulos** para cada coluna.
- O **tipo de dado (`Dtype`)** de cada coluna.
- O uso de memória.

**O que procurar aqui?**

1.  **Contagem de Não Nulos:** Se o valor for menor que o total de entradas, significa que a coluna tem dados faltantes.
2.  **Dtype:** O tipo de dado está correto? Uma coluna de valor de compra deveria ser numérica (`float64` ou `int64`), não `object` (que geralmente significa string). Uma coluna de data deveria ser `datetime64[ns]`, não `object`.


In [4]:
# Obtendo um resumo técnico do DataFrame
df_usuarios.info()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   user_id              215 non-null    str  
 1   nome                 215 non-null    str  
 2   email                189 non-null    str  
 3   data_cadastro        215 non-null    str  
 4   cidade               215 non-null    str  
 5   estado               215 non-null    str  
 6   valor_ultima_compra  192 non-null    str  
 7   data_ultimo_login    215 non-null    str  
dtypes: str(8)
memory usage: 41.5 KB


#### Usando `.describe(include='all')` para Estatísticas Descritivas

Enquanto `.info()` nos dá a estrutura, `.describe()` nos dá um resumo estatístico. Usando `include='all'`, forçamos o Pandas a nos mostrar estatísticas tanto para colunas numéricas quanto para as de texto (categóricas).

- **Para colunas numéricas:** `count`, `mean` (média), `std` (desvio padrão), `min`, `max`, e os quartis (`25%`, `50%`, `75%`).
- **Para colunas de objeto/categóricas:** `count`, `unique` (número de valores únicos), `top` (valor mais frequente), e `freq` (frequência do valor mais frequente).


In [5]:
# Obtendo um resumo estatístico de todas as colunas
df_usuarios.describe(include='all')

,user_id,nome,email,data_cadastro,cidade,estado,valor_ultima_compra,data_ultimo_login
count,215,215,189,215,215,215,192,215
unique,200,198,176,192,161,10,168,200
top,8df862a8-e353-4f29-8697-918fa854a6a0,Heloisa Barros,ravi-lucca32@example.net,09-05-2024,Souza,PR,Não informado,2026-08-01 10:48:34.582515
freq,2,2,2,3,4,36,10,2


### ✏️ Exercício 1: Análise Pós-Inspeção

Com base nas saídas dos comandos `.info()` e `.describe(include='all')`, responda na célula abaixo às seguintes perguntas:

1.  Quais colunas têm valores faltantes? A contagem de não nulos em `.info()` te deu essa resposta.
2.  A coluna `valor_ultima_compra` é do tipo correto para realizarmos cálculos (como a média)?
3.  As colunas `data_cadastro` e `data_ultimo_login` estão em um formato de data que o pandas entende nativamente?
4.  Olhando para a estatística `unique` da coluna `estado` no `.describe()`, o número parece alto ou baixo demais? O que isso pode indicar?


Responda aqui.


### 2.3 Tratando Dados Faltantes (Valores Nulos)

Nossa investigação revelou que as colunas `email` e `valor_ultima_compra` têm valores faltantes. Vamos lidar com eles.


In [6]:
# Contando o número de valores nulos em cada coluna
df_usuarios.isnull().sum()

user_id                 0
nome                    0
email                  26
data_cadastro           0
cidade                  0
estado                  0
valor_ultima_compra    23
data_ultimo_login       0
dtype: int64

#### Estratégias para Lidar com Dados Faltantes

Existem duas estratégias principais para lidar com dados faltantes:

1.  **Remoção:** Excluir as linhas (ou colunas) que contêm valores faltantes. É uma abordagem rápida e simples, mas tem um custo: a perda de dados. Se uma linha tem um valor importante faltando, mas as outras informações são valiosas, removê-la pode não ser o ideal.
2.  **Imputação (Preenchimento):** Preencher os valores faltantes com um valor estimado. Pode ser um valor fixo (como 0), a média, a mediana ou a moda da coluna. Esta abordagem preserva o resto dos dados da linha, mas introduz um valor que não é original.

A escolha da estratégia depende do contexto do negócio e da natureza da coluna.


#### Estratégia 1: Remover Linhas com `dropna()`

**Cenário:** Um usuário sem e-mail é de pouca utilidade para nosso e-commerce. Não podemos contatá-lo para marketing, recuperação de senha ou confirmação de pedidos. Portanto, a decisão de negócio aqui é **remover** os cadastros que não possuem um e-mail.

Usaremos `df.dropna(subset=['nome_da_coluna'])` para remover apenas as linhas onde o valor na coluna especificada é nulo.


In [7]:
# Verificando o número de linhas antes da remoção
print(
    f"Número de linhas antes de remover nulos em 'email': {len(df_usuarios)}")

# Contando os nulos em 'email' para confirmar
print(
    f"Número de valores nulos em 'email': {df_usuarios['email'].isnull().sum()}\n")

# Removendo as linhas onde a coluna 'email' é nula
# O parâmetro 'inplace=True' modifica o DataFrame diretamente, sem precisar de reatribuição (df = df.dropna(...))
df_usuarios.dropna(subset=['email'], inplace=True)

# Verificando o número de linhas depois da remoção
print(f"Número de linhas após remover nulos em 'email': {len(df_usuarios)}")

# Confirmando que não há mais nulos em 'email'
print(
    f"Número de valores nulos em 'email' agora: {df_usuarios['email'].isnull().sum()}")

Número de linhas antes de remover nulos em 'email': 215
Número de valores nulos em 'email': 26

Número de linhas após remover nulos em 'email': 189
Número de valores nulos em 'email' agora: 0


#### Estratégia 2: Imputar Valores com `fillna()`

**Cenário:** A coluna `valor_ultima_compra` também tem valores nulos. No entanto, remover essas linhas significaria perder informações de usuários que, embora não tenham um valor de compra registrado, ainda são clientes cadastrados. Uma abordagem melhor é a **imputação**.

**Média vs. Mediana:** Qual valor usar para preencher?

- **Média (`mean`):** A soma de todos os valores dividida pelo número de valores. É muito sensível a _outliers_ (valores extremamente altos ou baixos). Uma única compra de valor muito alto poderia inflar a média e distorcer a realidade.
- **Mediana (`median`):** O valor do meio quando todos os dados são ordenados. É robusta a outliers e, por isso, é geralmente a escolha mais segura para dados financeiros ou com distribuição assimétrica, como valores de compra.

Vamos tentar calcular a mediana e preencher os valores nulos. Mas... há um problema!


In [8]:
try:
    mediana_compra = df_usuarios['valor_ultima_compra'].median()
    print(f"Mediana calculada: {mediana_compra}")
except TypeError as e:
    print(f"Ocorreu um erro: {e}")
    print("\nNão podemos calcular a mediana de uma coluna que não é numérica! Isso nos leva ao próximo passo.")

Ocorreu um erro: Cannot perform reduction 'median' with string dtype

Não podemos calcular a mediana de uma coluna que não é numérica! Isso nos leva ao próximo passo.


### 2.4 Corrigindo Tipos de Dados

O erro acima aconteceu porque, como vimos no `.info()`, a coluna `valor_ultima_compra` é do tipo `object` (string), e não um número. Precisamos convertê-la.


#### Convertendo `valor_ultima_compra` para Numérico

Para converter, precisamos primeiro limpar a string, removendo o `R$ ` e trocando a vírgula decimal por um ponto. Depois, usamos `pd.to_numeric`.

O parâmetro `errors='coerce'` é muito útil: ele transformará qualquer valor que não possa ser convertido em um número (como a string 'Não informado') em `NaN`. Isso é ótimo, pois podemos tratar todos os problemas de uma vez só.


In [9]:
# Passo 1: Limpar a string
# Usamos.str para aplicar métodos de string a toda a coluna
df_usuarios['valor_ultima_compra'] = df_usuarios['valor_ultima_compra'].str.replace(
    'R$ ', '', regex=False)
df_usuarios['valor_ultima_compra'] = df_usuarios['valor_ultima_compra'].str.replace(
    ',', '.', regex=False)

# Passo 2: Converter para tipo numérico, tratando erros
df_usuarios['valor_ultima_compra'] = pd.to_numeric(
    df_usuarios['valor_ultima_compra'], errors='coerce')

# Vamos verificar o tipo de dado da coluna agora
print("Tipo de dado de 'valor_ultima_compra' após conversão:")
print(df_usuarios.dtypes['valor_ultima_compra'])

# E ver como ficaram os 10 primeiros valores
print("\nValores após conversão (note os novos NaNs onde antes era 'Não informado'):")
display(df_usuarios[['nome', 'valor_ultima_compra']].head(10))

Tipo de dado de 'valor_ultima_compra' após conversão:
float64

Valores após conversão (note os novos NaNs onde antes era 'Não informado'):


,nome,valor_ultima_compra
0,Ana Liz Novaes,968.82
1,Francisco Silveira,275.88
3,Sabrina Abreu,754.25
5,Otto Porto,NaN
6,Liz Aparecida,855.67
7,Maria Vitória Borges,622.53
8,Dr. Lorenzo Costela,328.14
9,Dr. Vicente Abreu,148.21
10,Caleb Sousa,206.54
11,Mariah Porto,913.75


#### Agora sim: Imputando a Mediana

Com a coluna `valor_ultima_compra` agora no formato `float64`, podemos finalmente calcular a mediana e usar `fillna()` para preencher todos os valores `NaN` (os que já existiam e os que foram criados pelo `errors='coerce'`).


In [10]:
# Verificando nulos ANTES da imputação
print(
    f"Nulos em 'valor_ultima_compra' ANTES da imputação: {df_usuarios['valor_ultima_compra'].isnull().sum()}")

# 1. Calcular a mediana (agora vai funcionar!)
mediana_compra = df_usuarios['valor_ultima_compra'].median()
print(f"A mediana calculada é: R$ {mediana_compra:.2f}")

# 2. Preencher os valores nulos com a mediana
df_usuarios['valor_ultima_compra'].fillna(mediana_compra, inplace=True)

# Verificando nulos depois da imputação
print(
    f"Nulos em 'valor_ultima_compra' APÓS a imputação: {df_usuarios['valor_ultima_compra'].isnull().sum()}")

Nulos em 'valor_ultima_compra' ANTES da imputação: 31
A mediana calculada é: R$ 457.25
Nulos em 'valor_ultima_compra' APÓS a imputação: 31


/tmp/ipykernel_132040/611049897.py:10: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_usuarios['valor_ultima_compra'].fillna(mediana_compra, inplace=True)


#### Convertendo Colunas de Data

As colunas `data_cadastro` e `data_ultimo_login` também são do tipo `object`. Precisamos convertê-las para o tipo `datetime` para que possamos realizar operações com datas, como calcular a quanto tempo um usuário se cadastrou.

A função `pd.to_datetime` é extremamente poderosa. Para a `data_cadastro`, que tem vários formatos, podemos usar o argumento `format='mixed'` para que o Pandas tente adivinhar o formato correto para cada linha.

Novamente, usaremos `errors='coerce'` para converter qualquer data que não possa ser entendida em `NaT` (Not a Time), o equivalente a `NaN` para datas.


In [11]:
# Convertendo a coluna 'data_cadastro' para datetime
# 'format="mixed"' permite que o pandas tente adivinhar múltiplos formatos
df_usuarios['data_cadastro'] = pd.to_datetime(
    df_usuarios['data_cadastro'], format='mixed', errors='coerce', dayfirst=False)

# A coluna 'data_ultimo_login' tem um formato mais consistente, mas ainda é object
df_usuarios['data_ultimo_login'] = pd.to_datetime(
    df_usuarios['data_ultimo_login'], errors='coerce')

# Vamos verificar os tipos de dados novamente com.info()
print("--- Verificação dos Dtypes após conversão de datas ---")
df_usuarios.info()

--- Verificação dos Dtypes após conversão de datas ---
<class 'pandas.DataFrame'>
Index: 189 entries, 0 to 214
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   user_id              189 non-null    str           
 1   nome                 189 non-null    str           
 2   email                189 non-null    str           
 3   data_cadastro        189 non-null    datetime64[us]
 4   cidade               189 non-null    str           
 5   estado               189 non-null    str           
 6   valor_ultima_compra  158 non-null    float64       
 7   data_ultimo_login    189 non-null    datetime64[us]
dtypes: datetime64[us](2), float64(1), str(5)
memory usage: 30.3 KB


Sucesso! As colunas `valor_ultima_compra`, `data_cadastro` e `data_ultimo_login` agora têm os tipos de dados corretos (`float64` e `datetime64[ns]`), como esperado. Agora podemos fazer operações matemáticas e de data, como calcular o tempo desde o último login ou a média de compras.


### ✏️ Exercício 2: Cálculos Pós-Conversão

Agora que os tipos de dados estão corretos, realize as seguintes tarefas em células de código separadas:

1.  Calcule o valor **médio** da coluna `valor_ultima_compra` e imprima o resultado formatado.
2.  Encontre a data do **último login mais recente** em todo o dataset .
3.  Calcule há quantos dias foi o último login do **primeiro usuário** do DataFrame.


In [12]:
# 1. Calcule o valor médio da coluna 'valor_ultima_compra'

In [13]:
# 2. Encontre a data do último login mais recente

In [14]:
# 3. Calcule há quantos dias foi o último login do primeiro usuário

---


### 2.5 Removendo Duplicatas

Dados duplicados podem distorcer análises, como a contagem de usuários únicos. Vamos verificar se existem e removê-los.

- `.duplicated().sum()`: Conta quantas linhas são duplicatas exatas de outras que já apareceram.
- `.drop_duplicates()`: Retorna um DataFrame com as duplicatas removidas.


In [15]:
# Verificando o número de linhas duplicadas
num_duplicatas = df_usuarios.duplicated().sum()
print(f"Número de linhas duplicadas encontradas: {num_duplicatas}")

# Removendo as duplicatas
print(f"Linhas antes de remover duplicatas: {len(df_usuarios)}")
df_usuarios.drop_duplicates(inplace=True)
print(f"Linhas após remover duplicatas: {len(df_usuarios)}")

Número de linhas duplicadas encontradas: 13
Linhas antes de remover duplicatas: 189
Linhas após remover duplicatas: 176


### 2.6 Padronizando Dados Categóricos

O último passo da nossa limpeza é garantir que dados de texto (categóricos) sejam consistentes. Na nossa inspeção, suspeitamos da coluna `estado`.

Vamos usar `.unique()` para ver todos os valores distintos que a coluna possui.


In [16]:
# Verificando os valores únicos na coluna 'estado'
print("Valores únicos em 'estado' ANTES da padronização:")
print(df_usuarios['estado'].unique())

# Criando o dicionário de mapeamento para corrigir as inconsistências
mapa_estados = {
    'São Paulo': 'SP',
    'sao paulo': 'SP',
    'Rio de Janeiro': 'RJ',
    'rio de janeiro': 'RJ'
    # Não precisamos mapear 'SP' -> 'SP' ou 'RJ' -> 'RJ', o replace ignora chaves que não encontra
}

# Aplicando a substituição
df_usuarios['estado'].replace(mapa_estados, inplace=True)

# Verificando os valores únicos novamente para confirmar a limpeza
print("\nValores únicos em 'estado' APÓS a padronização:")
print(df_usuarios['estado'].unique())

Valores únicos em 'estado' ANTES da padronização:
<ArrowStringArray>
[     'sao paulo',             'PR',             'SC',             'BA',
 'rio de janeiro',      'São Paulo', 'Rio de Janeiro',             'MG',
             'SP',             'RJ']
Length: 10, dtype: str

Valores únicos em 'estado' APÓS a padronização:
<ArrowStringArray>
[     'sao paulo',             'PR',             'SC',             'BA',
 'rio de janeiro',      'São Paulo', 'Rio de Janeiro',             'MG',
             'SP',             'RJ']
Length: 10, dtype: str


/tmp/ipykernel_132040/3736765885.py:15: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_usuarios['estado'].replace(mapa_estados, inplace=True)


Excelente! Agora nossa coluna `estado` está limpa e padronizada, pronta para ser usada em análises de agrupamento (`groupby`).


### 2.7 Salvando o Trabalho

Missão cumprida! Passamos por todas as etapas do nosso fluxo de trabalho de limpeza. O passo final é salvar nosso DataFrame limpo em um novo arquivo CSV. Este arquivo será o ponto de partida para futuras análises.


Vamos dar uma última olhada no nosso trabalho com `.info()` para confirmar que tudo está em ordem: sem nulos e com os tipos de dados corretos.


In [17]:
# Verificação final
df_usuarios.info()

<class 'pandas.DataFrame'>
Index: 176 entries, 0 to 199
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   user_id              176 non-null    str           
 1   nome                 176 non-null    str           
 2   email                176 non-null    str           
 3   data_cadastro        176 non-null    datetime64[us]
 4   cidade               176 non-null    str           
 5   estado               176 non-null    str           
 6   valor_ultima_compra  145 non-null    float64       
 7   data_ultimo_login    176 non-null    datetime64[us]
dtypes: datetime64[us](2), float64(1), str(5)
memory usage: 28.2 KB


In [18]:
# Salvando o DataFrame limpo em um novo arquivo CSV
df_usuarios.to_csv('usuarios_limpo.csv', index=False)

print("Arquivo 'usuarios_limpo.csv' salvo com sucesso!")

Arquivo 'usuarios_limpo.csv' salvo com sucesso!


---

## Conclusão

Parabéns! Você completou um ciclo completo de limpeza de dados. Você pegou um dataset caótico e, aplicando um método sistemático, o transformou em uma fonte de dados organizada e confiável.

**O que nós fizemos:**

- **Inspecionamos** os dados para encontrar problemas.
- **Tratamos valores faltantes** usando duas estratégias diferentes (remoção e imputação).
- **Corrigimos tipos de dados** incorretos, permitindo cálculos e operações.
- **Removemos dados duplicados** para garantir a unicidade dos registros.
- **Padronizamos dados categóricos** para permitir agrupamentos e análises consistentes.

Agora, com o arquivo `usuarios_limpo.csv` em mãos, você está pronto para a próxima aula, onde vamos explorar e visualizar esses dados para descobrir padrões e insights de negócio.
